# NYC, NY parcels

Build a full NYC export from NYC MapPLUTO (all 5 boroughs) via ArcGIS FeatureServer.

**Workflow summary**
1. Download all MapPLUTO parcels via paginated POST queries (or load cached parquet).
2. Filter out fully tax-exempt parcels (AssessTot == 0 AND AssessLand == 0).
3. Collapse condo unit parcels (Lot >= 1001) by Borough+Block, summing numeric fields and unioning geometries.
4. Categorize property types using LandUse codes and BldgClass.
5. Compute value fields, improvement ratios, and refined categories.
6. Export to GeoParquet (EPSG:4326) and upload to Azure dev blob.
7. Generate PMTiles via parquet_to_pmtiles.py and upload to dev.

In [1]:
import glob
import json
import os
import sys
from datetime import datetime
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import requests
from shapely.ops import unary_union

sys.path.append("..")
from parcel_calculations import add_improvement_ratio_fields

# Config
SCRAPE_DATA = 1  # set to 1 to pull a fresh scrape from MapPLUTO, 0 to load cached parquet
DATA_DIR = "data/nyc"
os.makedirs(DATA_DIR, exist_ok=True)

SERVICE_URL = (
    "https://services5.arcgis.com/GfwWNkhOj9bNBqoJ/arcgis/rest/services/"
    "MAPPLUTO/FeatureServer/0/query"
)

# TaxClass is not available in this service; use ExemptTot for exemption detection
OUT_FIELDS = [
    "OBJECTID",
    "BBL",
    "Borough",
    "Block",
    "Lot",
    "Address",
    "OwnerName",
    "LandUse",
    "BldgClass",
    "LotArea",
    "BldgArea",
    "AssessLand",
    "AssessTot",
    "ExemptTot",   # Total exempt value — used to identify fully exempt parcels
    "YearBuilt",
    "NumFloors",
    "UnitsRes",
    "UnitsTotal",
]

print(f"DATA_DIR: {DATA_DIR}")
print(f"SCRAPE_DATA: {SCRAPE_DATA}")

DATA_DIR: data/nyc
SCRAPE_DATA: 1


In [2]:
def get_total_count():
    """Get total record count from the MapPLUTO FeatureServer."""
    params = {
        "where": "1=1",
        "returnCountOnly": "true",
        "f": "json",
    }
    response = requests.post(SERVICE_URL, data=params, timeout=60)
    response.raise_for_status()
    payload = response.json()
    return payload.get("count", 0)


def query_mappluto_all(out_fields, batch_size=1000):
    """Download all NYC MapPLUTO parcels via paginated POST requests.
    
    Uses OBJECTID-range batching to avoid server transfer limits.
    """
    from shapely.geometry import shape as shapely_shape

    # First, get the total count and OBJECTID range
    print("  Getting total record count...")
    total = get_total_count()
    print(f"  Total records: {total:,}")

    # Get min/max OBJECTID to enable range batching
    params_stats = {
        "where": "1=1",
        "outStatistics": '[{"statisticType":"min","onStatisticField":"OBJECTID","outStatisticFieldName":"min_oid"},{"statisticType":"max","onStatisticField":"OBJECTID","outStatisticFieldName":"max_oid"}]',
        "f": "json",
    }
    resp = requests.post(SERVICE_URL, data=params_stats, timeout=60)
    resp.raise_for_status()
    stats = resp.json()
    min_oid = stats["features"][0]["attributes"]["min_oid"]
    max_oid = stats["features"][0]["attributes"]["max_oid"]
    print(f"  OBJECTID range: {min_oid} to {max_oid}")

    all_rows = []
    current_oid = min_oid

    while current_oid <= max_oid:
        end_oid = current_oid + batch_size - 1
        where_clause = f"OBJECTID >= {current_oid} AND OBJECTID <= {end_oid}"

        params = {
            "where": where_clause,
            "outFields": ",".join(out_fields),
            "outSR": 4326,
            "returnGeometry": "true",
            "f": "geojson",
            "resultRecordCount": batch_size + 100,  # slight buffer
        }
        response = requests.post(SERVICE_URL, data=params, timeout=120)
        response.raise_for_status()
        payload = response.json()
        features = payload.get("features", [])

        for feat in features:
            props = feat.get("properties") or {}
            geom_json = feat.get("geometry")
            geom = None
            if geom_json:
                try:
                    geom = shapely_shape(geom_json)
                except Exception:
                    geom = None
            row = dict(props)
            row["geometry"] = geom
            all_rows.append(row)

        current_oid = end_oid + 1

        if len(all_rows) % 20000 < batch_size:
            print(f"  Fetched {len(all_rows):,} records so far (OBJECTID up to {end_oid})...")

    print(f"✅ Total records downloaded: {len(all_rows):,}")

    if not all_rows:
        raise RuntimeError("No records downloaded — check API connectivity")

    df = pd.DataFrame(all_rows)
    geom_series = gpd.GeoSeries(df["geometry"].values, crs="EPSG:4326")
    df = df.drop(columns=["geometry"])
    gdf = gpd.GeoDataFrame(df, geometry=geom_series, crs="EPSG:4326")

    # Drop rows with null geometry
    null_geom = gdf.geometry.isna()
    if null_geom.sum() > 0:
        print(f"  Dropping {null_geom.sum():,} rows with null geometry")
        gdf = gdf[~null_geom].copy()

    return gdf


if SCRAPE_DATA == 1:
    print("🔄 Downloading full MapPLUTO dataset (all NYC boroughs)...")
    print("   This will use OBJECTID-range batching to avoid server limits.")
    parcels_gdf = query_mappluto_all(OUT_FIELDS)

    today_str = datetime.now().strftime("%Y_%m_%d")
    raw_path = os.path.join(DATA_DIR, f"mappluto_raw_{today_str}.parquet")
    parcels_gdf.to_parquet(raw_path, index=False)
    print(f"✅ Saved raw scrape to {raw_path}")
else:
    files = glob.glob(os.path.join(DATA_DIR, "mappluto_raw_*.parquet"))
    if not files:
        raise FileNotFoundError(
            f"No raw parquet files found in {DATA_DIR}. Set SCRAPE_DATA=1 to scrape."
        )
    files_sorted = sorted(
        files,
        key=lambda x: datetime.strptime(
            os.path.basename(x).replace("mappluto_raw_", "").replace(".parquet", ""),
            "%Y_%m_%d",
        ),
        reverse=True,
    )
    latest_file = files_sorted[0]
    print(f"✅ Loading most recent scrape: {latest_file}")
    parcels_gdf = gpd.read_parquet(latest_file)

print(f"✅ Loaded {len(parcels_gdf):,} rows | CRS={parcels_gdf.crs}")

🔄 Downloading full MapPLUTO dataset (all NYC boroughs)...
   This will use OBJECTID-range batching to avoid server limits.
  Getting total record count...


  Total records: 856,670


  OBJECTID range: 1 to 856670


  Fetched 20,000 records so far (OBJECTID up to 20000)...


  Fetched 40,000 records so far (OBJECTID up to 40000)...


...

  Fetched 800,000 records so far (OBJECTID up to 800000)...


  Fetched 820,000 records so far (OBJECTID up to 820000)...


  Fetched 840,000 records so far (OBJECTID up to 840000)...


✅ Total records downloaded: 856,670


✅ Saved raw scrape to data/nyc/mappluto_raw_2026_02_18.parquet
✅ Loaded 856,670 rows | CRS=EPSG:4326


In [3]:
# Filter out fully tax-exempt parcels
# In MapPLUTO, ExemptTot is the total exempt assessed value.
# A parcel is fully exempt when ExemptTot >= AssessTot (100% of value is exempt),
# OR when both AssessTot == 0 and AssessLand == 0 (no assessed value at all).
parcels_gdf["AssessTot"] = pd.to_numeric(parcels_gdf["AssessTot"], errors="coerce").fillna(0)
parcels_gdf["AssessLand"] = pd.to_numeric(parcels_gdf["AssessLand"], errors="coerce").fillna(0)
parcels_gdf["ExemptTot"] = pd.to_numeric(parcels_gdf["ExemptTot"], errors="coerce").fillna(0)

parcels_gdf["fully_exempt"] = (
    (parcels_gdf["ExemptTot"] >= parcels_gdf["AssessTot"]) & (parcels_gdf["AssessTot"] > 0)
) | (
    (parcels_gdf["AssessTot"] == 0) & (parcels_gdf["AssessLand"] == 0)
)

before_count = len(parcels_gdf)
parcels_gdf = parcels_gdf[~parcels_gdf["fully_exempt"]].copy()
after_count = len(parcels_gdf)

print(f"✅ Removed {before_count - after_count:,} fully exempt parcels")
print(f"✅ Rows remaining: {after_count:,}")

✅ Removed 33,452 fully exempt parcels
✅ Rows remaining: 823,218


In [4]:
# Condo deduplication
# In NYC MapPLUTO, condo units have Lot >= 1001. We collapse them by Borough+Block,
# summing numeric fields and unioning geometries, so each condo building appears once.

def is_condo_lot(lot_val):
    try:
        return int(str(lot_val).strip()) >= 1001
    except (ValueError, TypeError):
        return False

parcels_gdf["is_condo_unit"] = parcels_gdf["Lot"].apply(is_condo_lot)
condo_count = parcels_gdf["is_condo_unit"].sum()
print(f"Condo unit parcels (Lot >= 1001): {condo_count:,}")

# Assign group key: condo units collapse by Borough+Block; non-condos stay unique by BBL
parcels_gdf["parcel_group_id"] = parcels_gdf.apply(
    lambda r: f"{r['Borough']}_{r['Block']}_condo" if r["is_condo_unit"] else str(r["BBL"]),
    axis=1,
)

dup_count = parcels_gdf.duplicated(subset=["parcel_group_id"], keep=False).sum()
print(f"Duplicate rows by parcel_group_id before collapse: {dup_count:,}")

numeric_sum_cols = [
    "AssessLand", "AssessTot", "UnitsRes", "UnitsTotal", "BldgArea", "LotArea",
]
numeric_sum_cols = [c for c in numeric_sum_cols if c in parcels_gdf.columns]

categorical_cols = [
    c for c in parcels_gdf.columns
    if c not in set(numeric_sum_cols + ["geometry", "parcel_group_id"])
]

agg_dict = {c: "sum" for c in numeric_sum_cols}
agg_dict.update({c: "first" for c in categorical_cols})

collapsed = (
    parcels_gdf.groupby("parcel_group_id", dropna=False)
    .agg(agg_dict)
    .reset_index(drop=True)
)

# Union geometries per group
geom_union = parcels_gdf.groupby("parcel_group_id", dropna=False)["geometry"].apply(
    lambda geoms: unary_union([g for g in geoms if g is not None])
    if any(g is not None for g in geoms)
    else None
)
collapsed["geometry"] = geom_union.values

parcels_gdf = gpd.GeoDataFrame(collapsed, geometry="geometry", crs="EPSG:4326")
print(f"✅ Rows after condo collapse: {len(parcels_gdf):,}")

Condo unit parcels (Lot >= 1001): 12,803


Duplicate rows by parcel_group_id before collapse: 10,324


✅ Rows after condo collapse: 815,355


In [5]:
# Property type categorization
# Uses MapPLUTO LandUse codes (2-digit strings) and BldgClass prefix for vacant detection

LAND_USE_MAP = {
    "01": "Single-Family",
    "02": "Two-Family",
    "03": "Three-Family",
    "04": "Multi-Family",
    "05": "Mixed Residential",
    "06": "Commercial",
    "07": "Commercial",
    "08": "Industrial",
    "09": "Parking Lot",
    "10": "Public Facilities",
    "11": "Public Facilities",
}


def categorize_property_type(land_use, bldg_class):
    bldg_class = str(bldg_class) if pd.notna(bldg_class) else ""
    # V-prefix building classes are vacant lots
    if bldg_class.startswith("V"):
        return "Vacant"
    land_use = str(land_use).strip().zfill(2) if pd.notna(land_use) else ""
    return LAND_USE_MAP.get(land_use, "Other")


parcels_gdf["property_land_use_category"] = parcels_gdf.apply(
    lambda r: categorize_property_type(r.get("LandUse"), r.get("BldgClass")), axis=1
)

print("Property category distribution:")
print(parcels_gdf["property_land_use_category"].value_counts().to_string())

Property category distribution:
property_land_use_category
Single-Family        563770
Two-Family           125642
Multi-Family          53028
Mixed Residential     20277
Vacant                16584
Commercial            12357
Three-Family          10564
Public Facilities      7883
Industrial             3051
Other                  1783
Parking Lot             416


In [6]:
export_gdf = parcels_gdf.copy()

# Core value fields
# MapPLUTO AssessLand = assessed land value; AssessTot = total assessed (land + improvement)
export_gdf["land_value"] = pd.to_numeric(export_gdf["AssessLand"], errors="coerce")
export_gdf["full_market_value"] = pd.to_numeric(export_gdf["AssessTot"], errors="coerce")
export_gdf["improvement_value"] = export_gdf["full_market_value"] - export_gdf["land_value"]
export_gdf["improvement_value"] = export_gdf["improvement_value"].clip(lower=0)  # no negatives

# REALLANDVA / REALIMPROV aliases required by parquet_to_pmtiles.py compute_metadata()
export_gdf["REALLANDVA"] = export_gdf["land_value"]
export_gdf["REALIMPROV"] = export_gdf["improvement_value"]

# Area in square feet (LotArea is already in sqft for MapPLUTO)
export_gdf["area_sqft"] = pd.to_numeric(export_gdf["LotArea"], errors="coerce").replace(0, np.nan)

# Per-sqft fields
export_gdf["full_market_value_per_sqft"] = export_gdf["full_market_value"] / export_gdf["area_sqft"]
export_gdf["land_value_per_sqft"] = export_gdf["land_value"] / export_gdf["area_sqft"]
export_gdf["improvement_value_per_sqft"] = export_gdf["improvement_value"] / export_gdf["area_sqft"]
export_gdf["REALLANDVA_per_sqft"] = export_gdf["REALLANDVA"] / export_gdf["area_sqft"]
export_gdf["REALIMPROV_per_sqft"] = export_gdf["REALIMPROV"] / export_gdf["area_sqft"]
export_gdf["TLLDIMPROV_per_sqft"] = export_gdf["full_market_value"] / export_gdf["area_sqft"]

# Exemption flag: all remaining parcels are non-exempt (we already filtered fully exempt above)
export_gdf["exemption_flag"] = 0

# Refined property category
def categorize_property_refined(row):
    cat = str(row.get("property_land_use_category") or "")
    if "Vacant" in cat:
        return "Vacant"
    if "Parking" in cat:
        return "Parking Lot"
    lv = row.get("land_value") or 0
    iv = row.get("improvement_value") or 0
    if pd.notna(iv) and pd.notna(lv) and (lv + iv) > 0 and iv < 0.5 * (lv + iv):
        return "Underdeveloped"
    return None


export_gdf["property_land_use_refined"] = export_gdf.apply(categorize_property_refined, axis=1)

# Improvement ratio fields
export_gdf = add_improvement_ratio_fields(
    export_gdf,
    land_col="land_value",
    improvement_col="improvement_value",
)

# Property lookup link (ACRIS BBL search)
export_gdf["link"] = (
    "https://a836-acris.nyc.gov/bblsearch/bblsearch.asp?borough="
    + export_gdf["Borough"].astype(str)
    + "&block="
    + export_gdf["Block"].astype(str)
    + "&lot="
    + export_gdf["Lot"].astype(str)
)

print("Refined category distribution:")
print(export_gdf["property_land_use_refined"].value_counts(dropna=False).to_string())

Refined category distribution:
property_land_use_refined
None              781907
Vacant             16584
Underdeveloped     16448
Parking Lot          416


In [7]:
columns_to_export = [
    "geometry",
    "exemption_flag",
    "property_land_use_category",
    "property_land_use_refined",
    "full_market_value",
    "full_market_value_per_sqft",
    "land_value",
    "land_value_per_sqft",
    "improvement_value",
    "improvement_value_per_sqft",
    "REALLANDVA",
    "REALIMPROV",
    "REALLANDVA_per_sqft",
    "REALIMPROV_per_sqft",
    "TLLDIMPROV_per_sqft",
    "TLLDIMPROV",
    "IMPR_LAND_RATIO",
    "IMPR_LAND_PCT",
    "IMPR_PCT_TOTAL",
    "link",
]

for col in columns_to_export:
    if col not in export_gdf.columns:
        export_gdf[col] = np.nan

export_final = export_gdf[columns_to_export].rename(
    columns={"land_value": "current_full_land_value"}
)

# Fix any invalid geometries
export_final["geometry"] = export_final["geometry"].apply(
    lambda geom: geom if geom is None or geom.is_valid else geom.buffer(0)
)

export_final = gpd.GeoDataFrame(export_final, geometry="geometry", crs="EPSG:4326")
if export_final.crs is None or export_final.crs.to_epsg() != 4326:
    export_final = export_final.to_crs("EPSG:4326")

# Save to data/jurisidictions/data/nyc/ (matching Denver pattern for PMTiles script)
# Notebook runs from data/jurisidictions/, so resolve relative to this file's location
notebook_dir = Path(os.getcwd())
output_dir = notebook_dir / "data" / "nyc"
output_dir.mkdir(parents=True, exist_ok=True)

canonical_path = str(output_dir / "nyc-ny-parcels.parquet")
today_str = datetime.now().strftime("%Y_%m_%d")
dated_path = str(output_dir / f"nyc-ny-parcels_{today_str}.parquet")

export_final.to_parquet(canonical_path, index=False)
export_final.to_parquet(dated_path, index=False)

print(f"✅ Saved export parquet: {canonical_path}")
print(f"✅ Also saved dated version: {dated_path}")
print(f"Total rows exported: {len(export_final):,}")
print(export_final.dtypes)

✅ Saved export parquet: ./data/jurisidictions/data/nyc/nyc-ny-parcels.parquet
✅ Also saved dated version: ./data/jurisidictions/data/nyc/nyc-ny-parcels_2026_02_18.parquet
Total rows exported: 815,355
geometry                      geometry
exemption_flag                   int64
property_land_use_category      object
property_land_use_refined       object
full_market_value              float64
full_market_value_per_sqft     float64
current_full_land_value        float64
land_value_per_sqft            float64
improvement_value              float64
improvement_value_per_sqft     float64
REALLANDVA                     float64
REALIMPROV                     float64
REALLANDVA_per_sqft            float64
REALIMPROV_per_sqft            float64
TLLDIMPROV_per_sqft            float64
TLLDIMPROV                     float64
IMPR_LAND_RATIO                float64
IMPR_LAND_PCT                  float64
IMPR_PCT_TOTAL                 float64
link                            object
dtype: object


In [8]:
# Upload parquet to Azure Dev blob
upload_dev_parquet = True

if upload_dev_parquet:
    from azure.storage.blob import BlobServiceClient

    connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
    if not connection_string:
        raise ValueError(
            "Set AZURE_STORAGE_CONNECTION_STRING before upload."
        )

    container = os.getenv("AZURE_DEV_CONTAINER", "parquets-dev")
    blob_name = "nyc-ny-parcels.parquet"

    blob_service = BlobServiceClient.from_connection_string(connection_string)
    container_client = blob_service.get_container_client(container)

    with open(canonical_path, "rb") as handle:
        container_client.upload_blob(name=blob_name, data=handle, overwrite=True)

    print(f"✅ Uploaded {canonical_path} -> {container}/{blob_name}")
else:
    print("upload_dev_parquet is False; skipping parquet upload.")

✅ Uploaded ./data/jurisidictions/data/nyc/nyc-ny-parcels.parquet -> parquets-dev/nyc-ny-parcels.parquet


In [9]:
# Generate PMTiles and upload to Azure Dev
# Uses parquet_to_pmtiles.py (same pattern as Denver)
upload_dev_pmtiles = True

if upload_dev_pmtiles:
    import subprocess
    import sys
    from pathlib import Path

    # Find project root by locating data/scripts/parquet_to_pmtiles.py
    current = Path.cwd()
    project_root = None
    while current.parent != current:
        if (current / "data" / "scripts" / "parquet_to_pmtiles.py").exists():
            project_root = current
            break
        current = current.parent

    if project_root is None:
        raise RuntimeError("Could not find project root containing data/scripts/parquet_to_pmtiles.py")

    script_path = project_root / "data" / "scripts" / "parquet_to_pmtiles.py"
    print(f"Using script: {script_path}")
    print(f"Running from: {project_root}")

    cmd = [
        sys.executable,
        str(script_path),
        "--city", "nyc",
        "--upload",
        "--overwrite",
    ]

    result = subprocess.run(cmd, cwd=str(project_root), capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:", result.stderr)
        raise RuntimeError(f"parquet_to_pmtiles.py failed with exit code {result.returncode}")
    print("✅ PMTiles generation and upload complete!")
else:
    print("upload_dev_pmtiles is False; skipping PMTiles generation.")

Using script: ./data/scripts/parquet_to_pmtiles.py
Running from: .


Step 1: Loading parquet and computing metadata
Loaded 815,355 features
Computed metadata: 16 fields, 3 refined categories
✅ Metadata saved: data/jurisidictions/data/nyc/nyc-ny-parcels-metadata.json
Step 2: Checking dependencies
✅ All dependencies found
Step 3: Converting to PMTiles
Writing base GeoJSON
Building low-zoom aggregate layer
✅ Low-zoom aggregate layer created
Creating MBTiles: /var/folders/jb/s1bhbc2x3dgbhdvtbnz335sc0000gn/T/tmpzpmyj1tz/output.mbtiles
Running: tippecanoe -o /var/folders/jb/s1bhbc2x3dgbhdvtbnz335sc0000gn/T/tmpzpmyj1tz/output.mbtiles -z 14 -Z 0 --coalesce-densest-as-needed --extend-zooms-if-still-dropping -L parcels:/var/folders/jb/s1bhbc2x3dgbhdvtbnz335sc0000gn/T/tmpzpmyj1tz/input.geojson -L parcels_low:/var/folders/jb/s1bhbc2x3dgbhdvtbnz335sc0000gn/T/tmpzpmyj1tz/input_low_zoom.geojson
✅ MBTiles created: /var/folders/jb/s1bhbc2x3dgbhdvtbnz335sc0000gn/T/tmpzpmyj1tz/output.mbtiles
Converting MBTiles to PMTiles: data/jurisidictions/data/nyc/nyc-ny-parcels.pmtile